# RFT-0003A — Pro4 private-hint Qwen solution generation

This notebook does only train-data generation. It selects official-train
questions with prior native-Qwen success rate at most 50% (`<=2/4`, or
the reused pilot equivalent `<=4/8`), uses existing verified Solar-Pro4
solutions as private hints, and asks frozen Lane B checkpoint-178 to produce
four independent Qwen-style solutions. Only strict official-answer-matching
terminal `\boxed{INTEGER}` outputs survive. Pro4 text itself is never used
as the SFT response. Tune/dev/test holdout IDs and templates are read only
to exclude them from the generated training corpus.

In [ ]:
# Cell 1 — Install once in a fresh A100 runtime, then restart only if requested.
%pip install -q --no-cache-dir \
  "nvidia-cuda-runtime==13.0.88" \
  "nvidia-cuda-nvrtc==13.0.88" \
  "vllm==0.26.0" \
  "pandas>=2.2,<3"
%pip uninstall -q -y torchcodec
print("[SETUP] complete. If vLLM was already imported, restart once and continue at Cell 2.")

In [ ]:
# Cell 2 — Resolve train-only generation inputs and holdout IDs.
import ctypes
import difflib
import glob
import hashlib
import json
import logging
import math
import os
import re
import site
import subprocess
import sys
import time
import unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from google.colab import drive

os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
MODEL_REVISION = "main"
RUN_ID = "RFT-0003A-pro4-hint-qwen-generation"
R1_RUN_ID = "RFT-0002-r1-native-k4-full"
TRAIN_RUN_ID = "RFT-0002-r1-ab-bf16-lora-a100"
EVAL_RUN_ID = "RFT-0002-r1-ab-vllm-eval-a100"
PRO4_RUN_ID = "EXP-0007-upstage-pro4-train-cot-low4"
PILOT_PREFIX = "RFT-0001-phase1-pilot-"
SPLIT_RUN_ID = "AUDIT-0002-clean-split-passN-20260821-215706"
SELECTED_LANE = "lane_b_low_drift"
EXPECTED_CHECKPOINT = "checkpoint-178"
SEED = 20260825

N_SC = 16
SC_TEMPERATURE = 1.0
SC_TOP_P = 0.95
SC_MAX_NEW_TOKENS = 2048
N_RATIONALIZE = 4
RATIONALIZE_TEMPERATURE = 0.7
RATIONALIZE_TOP_P = 0.95
RATIONALIZE_MAX_NEW_TOKENS = 4096
MAX_MODEL_LEN = 8192
MAX_NUM_SEQS = 256
MAX_NUM_BATCHED_TOKENS = 65536
GPU_MEMORY_UTILIZATION = 0.94
TARGET_MAX_SUCCESS_RATE = 0.50
MAX_VISIBLE_WORDS = 350
MAX_SFT_TOKENS = 2048
TARGET_MIX_SHARE = 0.65

SYSTEM_PROMPT = "You are a helpful assistant that solves math problems step by step."
USER_SUFFIX = (
    "Solve this step by step, then give the final answer as a single integer "
    "inside \\boxed{}."
)
SC_PROMPT_VERSION = "champion_r1_boxed_k4_v1"
RATIONALIZE_PROMPT_VERSION = "r2_pro4_private_hint_qwen_rewrite_v1"

def compact(value):
    value = unicodedata.normalize("NFC", str(value)).casefold()
    return re.sub(r"[\s_\-]+", "", value)

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def read_csv_strings(path):
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    frame.columns = [column.strip() for column in frame.columns]
    return frame

mount = Path("/content/drive")
if not (mount / "MyDrive").exists():
    drive.mount(str(mount))
DRIVE_ROOT = mount / "MyDrive"
projects = [
    path for path in DRIVE_ROOT.iterdir()
    if path.is_dir() and compact(path.name) == compact("2026소중한챌린지")
]
assert len(projects) == 1, projects
PROJECT_DIR = projects[0]
RUNS_DIR = PROJECT_DIR / "runs"

data_dirs = [path for path in [PROJECT_DIR / "data", PROJECT_DIR / "Data"] if path.exists()]
assert data_dirs, "Project data directory not found."
clean_candidates = [path / "clean_train.csv" for path in data_dirs if (path / "clean_train.csv").exists()]
assert len(clean_candidates) == 1, clean_candidates
CLEAN_TRAIN_PATH = clean_candidates[0]

SPLIT_DIR = RUNS_DIR / SPLIT_RUN_ID / "splits"
TUNE_PATH = SPLIT_DIR / "tune_v1.csv"
DEV_PATH = SPLIT_DIR / "dev_v1.csv"
TEST_HOLDOUT_PATH = SPLIT_DIR / "test_v1.csv"
for path in [TUNE_PATH, DEV_PATH, TEST_HOLDOUT_PATH]:
    assert path.exists(), path

winners_path = RUNS_DIR / EVAL_RUN_ID / "reports" / "frozen_tune_winners.json"
winners = json.loads(winners_path.read_text(encoding="utf-8"))
ADAPTER_PATH = Path(winners[SELECTED_LANE]["path"])
assert ADAPTER_PATH.name == EXPECTED_CHECKPOINT, ADAPTER_PATH
assert (ADAPTER_PATH / "adapter_config.json").exists(), ADAPTER_PATH
adapter_weight = next(
    (ADAPTER_PATH / name for name in ["adapter_model.safetensors", "adapter_model.bin"]
     if (ADAPTER_PATH / name).exists()), None
)
assert adapter_weight is not None

R1_DIR = RUNS_DIR / R1_RUN_ID
R1_VERIFIED_PATH = R1_DIR / "data" / "r1_native_verified_full.csv"
R1_MANIFEST_PATH = R1_DIR / "reports" / "r1_native_manifest.json"
assert R1_VERIFIED_PATH.exists() and R1_MANIFEST_PATH.exists()

# The completed Pro4 corpus has appeared under both its original run
# directory and later uploaded teacher-core locations. Resolve by schema
# and row count instead of assuming one filename. Prefer a full
# upstage_pro4_verified_cot* artifact over a derived teacher_core* file.
pro4_candidate_specs = []
exact_pro4 = RUNS_DIR / PRO4_RUN_ID / "verified" / "upstage_pro4_verified_cot.csv"
if exact_pro4.exists():
    pro4_candidate_specs.append((0, exact_pro4))
for path in RUNS_DIR.glob("EXP-0007*/verified/upstage_pro4_verified_cot*.csv"):
    if path.exists():
        pro4_candidate_specs.append((0, path))
for data_dir in data_dirs:
    for path in data_dir.rglob("upstage_pro4_verified_cot*.csv"):
        pro4_candidate_specs.append((0, path))
    for path in data_dir.rglob("teacher_core*.csv"):
        pro4_candidate_specs.append((1, path))
for pattern, priority in [
    ("upstage_pro4_verified_cot*.csv", 0),
    ("teacher_core*.csv", 1),
]:
    for path in DRIVE_ROOT.glob(pattern):
        pro4_candidate_specs.append((priority, path))

valid_pro4_candidates = []
required_pro4_columns = {"id", "question", "answer", "solution"}
for priority, path in sorted(set(pro4_candidate_specs), key=lambda item: str(item[1])):
    try:
        probe = pd.read_csv(path, dtype=str, keep_default_na=False)
        probe.columns = [column.strip() for column in probe.columns]
    except Exception as exc:
        print("[PRO4 CANDIDATE REJECT]", path, repr(exc))
        continue
    if required_pro4_columns.issubset(probe.columns) and len(probe):
        valid_pro4_candidates.append((priority, len(probe), path))
        print("[PRO4 CANDIDATE]", path, "rows=", len(probe), "priority=", priority)

if not valid_pro4_candidates:
    raise FileNotFoundError(
        "No Pro4 training-hint CSV with columns id,question,answer,solution was found. "
        "Upload upstage_pro4_verified_cot.csv (preferred) or teacher_core_10k.csv "
        "under the project data/ directory, then rerun Cell 2."
    )
valid_pro4_candidates.sort(key=lambda item: (item[0], -item[1], str(item[2])))
_, PRO4_SOURCE_ROWS, PRO4_VERIFIED_PATH = valid_pro4_candidates[0]
print("[PRO4 SELECTED]", PRO4_VERIFIED_PATH, "rows=", PRO4_SOURCE_ROWS)

pilot_raw_candidates = sorted(
    RUNS_DIR.glob(f"{PILOT_PREFIX}*/candidates/pilot_2000_rollouts.jsonl")
)
assert pilot_raw_candidates, "Completed RFT-0001 pilot raw JSONL not found."
# Prefer the immutable completed 2K pilot with the strict dataset artifact.
pilot_valid = [
    path for path in pilot_raw_candidates
    if (path.parents[1] / "data" / "rft_pilot_strict.csv").exists()
]
assert pilot_valid, pilot_raw_candidates
PILOT_RAW_PATH = max(pilot_valid, key=lambda path: path.stat().st_size)
PILOT_RUN_DIR = PILOT_RAW_PATH.parents[1]

R1_RAW_PATHS = sorted((R1_DIR / "candidates").glob("r1_native_rollouts_shard_*.jsonl"))
assert len(R1_RAW_PATHS) == 3, R1_RAW_PATHS

label_review_candidates = sorted(
    PILOT_RUN_DIR.glob("data/label_review_queue.csv")
)
LABEL_REVIEW_PATH = label_review_candidates[0] if label_review_candidates else None

EXP_DIR = RUNS_DIR / RUN_ID
DATA_DIR = EXP_DIR / "data"
CAND_DIR = EXP_DIR / "candidates"
PRED_DIR = EXP_DIR / "predictions"
REPORT_DIR = EXP_DIR / "reports"
for path in [DATA_DIR, CAND_DIR, PRED_DIR, REPORT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

READ_PATHS = [
    CLEAN_TRAIN_PATH, TUNE_PATH, DEV_PATH, TEST_HOLDOUT_PATH,
    winners_path, R1_VERIFIED_PATH, R1_MANIFEST_PATH,
    PRO4_VERIFIED_PATH, PILOT_RAW_PATH, *R1_RAW_PATHS,
]
forbidden = [
    path for path in READ_PATHS
    if any(term in path.name.casefold() for term in ["leaderboard", "submission"])
    or path.name.casefold() in {"deep_chal_math_test.csv", "deep_chal_math_dataset_test.csv"}
]
assert not forbidden, f"Forbidden evaluation inputs: {forbidden}"
assert torch.cuda.is_available(), "GPU runtime required."
print("[PROJECT]", PROJECT_DIR)
print("[ADAPTER]", ADAPTER_PATH)
print("[ADAPTER SHA]", sha256_file(adapter_weight))
print("[PRO4 HINTS]", PRO4_VERIFIED_PATH)
print("[PILOT RAW]", PILOT_RAW_PATH)
print("[R1 RAW SHARDS]", len(R1_RAW_PATHS))
print("[SAFE] leaderboard/test/submission files are not read")

In [ ]:
# Cell 3 — Select <=50%-success train rows and verify existing Pro4 hints.
TERMINAL_BOX_RE = re.compile(r"\\boxed\s*\{\s*(-?\d(?:[\d,]*\d)?)\s*\}")

def normalize_integer(value):
    value = str(value or "").strip().replace(",", "")
    match = re.fullmatch(r"-?\d+(?:\.0+)?", value)
    return str(int(value.split(".")[0])) if match else None

def terminal_boxed(text):
    text = str(text or "")
    matches = list(TERMINAL_BOX_RE.finditer(text))
    if not matches:
        return None
    match = matches[-1]
    tail = text[match.end():].strip()
    while True:
        previous = tail
        tail = re.sub(r"^(?:\\\)|\\\]|\$\$|\$)", "", tail).strip()
        if tail == previous:
            break
    tail = re.sub(r"^[.!]+$", "", tail).strip()
    return normalize_integer(match.group(1)) if not tail else None

def normalize_question(text):
    text = unicodedata.normalize("NFKC", str(text)).casefold()
    return re.sub(r"\s+", " ", text).strip()

def template_key(text):
    text = normalize_question(text)
    return re.sub(r"\d+(?:\.\d+)?", "#", text)

def load_valid_jsonl(path):
    rows = []
    with Path(path).open(encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError as exc:
                print(f"[JSONL] skipped corrupt tail {path}:{line_number}: {exc}")
                continue
            if isinstance(row, dict) and row.get("id"):
                rows.append(row)
    return rows

clean_train = read_csv_strings(CLEAN_TRAIN_PATH)
tune = read_csv_strings(TUNE_PATH)
dev = read_csv_strings(DEV_PATH)
test_holdout = read_csv_strings(TEST_HOLDOUT_PATH)
for frame in [clean_train, tune, dev, test_holdout]:
    assert {"id", "question", "answer"}.issubset(frame.columns)
    frame["answer"] = frame["answer"].map(normalize_integer)
    assert frame["answer"].notna().all()

holdout_ids = set(tune["id"]) | set(dev["id"]) | set(test_holdout["id"])
holdout_templates = set(
    pd.concat([tune, dev, test_holdout], ignore_index=True)["question"].map(template_key)
)

success_rows = []
raw_paths = [PILOT_RAW_PATH, *R1_RAW_PATHS]
seen_ids = set()
for path in raw_paths:
    # Resume-safe JSONLs may contain an older duplicate ID. The latest
    # complete record in each file is authoritative, matching prior recovery.
    records_in_path = {
        str(row["id"]): row for row in load_valid_jsonl(path)
    }
    for row in records_in_path.values():
        qid = str(row["id"])
        assert qid not in seen_ids, f"Duplicate rollout ID across pilot/shards: {qid}"
        seen_ids.add(qid)
        official = normalize_integer(row.get("official_answer"))
        candidates = list(row.get("candidates", []))
        assert official is not None and len(candidates) in {4, 8}, (qid, len(candidates))
        strict_successes = sum(
            terminal_boxed(candidate.get("raw_output", "")) == official
            for candidate in candidates
        )
        success_rows.append({
            "id": qid,
            "official_answer_rollout": official,
            "strict_successes": strict_successes,
            "rollout_k": len(candidates),
            "strict_success_rate": strict_successes / len(candidates),
            "rollout_source": str(path),
        })

success = pd.DataFrame(success_rows)
assert success["id"].is_unique
label_review_ids = set()
if LABEL_REVIEW_PATH is not None and LABEL_REVIEW_PATH.exists():
    label_review = read_csv_strings(LABEL_REVIEW_PATH)
    label_review_ids = set(label_review["id"])

pro4 = read_csv_strings(PRO4_VERIFIED_PATH)
required_pro4 = {"id", "question", "answer", "solution"}
assert required_pro4.issubset(pro4.columns), pro4.columns.tolist()
pro4["answer"] = pro4["answer"].map(normalize_integer)
pro4 = pro4.drop_duplicates("id", keep="last")
pro4["pro4_terminal"] = pro4["solution"].map(terminal_boxed)
pro4["pro4_question_key"] = pro4["question"].map(normalize_question)

train = clean_train.copy()
train["question_key"] = train["question"].map(normalize_question)
train["template_key"] = train["question"].map(template_key)
train = train.merge(success, on="id", how="inner", validate="one_to_one")
train = train.merge(
    pro4[["id", "answer", "solution", "pro4_terminal", "pro4_question_key"]].rename(
        columns={"answer": "pro4_answer", "solution": "pro4_hint"}
    ),
    on="id", how="left", validate="one_to_one",
)

train["is_holdout"] = train["id"].isin(holdout_ids) | train["template_key"].isin(holdout_templates)
train["label_review"] = train["id"].isin(label_review_ids)
train["has_verified_pro4"] = (
    train["pro4_answer"].eq(train["answer"])
    & train["pro4_terminal"].eq(train["answer"])
    & train["pro4_question_key"].eq(train["question_key"])
)
train["target_difficulty"] = train["strict_success_rate"].le(TARGET_MAX_SUCCESS_RATE)

targets = train[
    train["target_difficulty"]
    & ~train["is_holdout"]
    & ~train["label_review"]
    & train["has_verified_pro4"]
].copy()
targets = targets.sort_values(["strict_success_rate", "id"]).reset_index(drop=True)
assert not set(targets["id"]) & holdout_ids
assert not set(targets["template_key"]) & holdout_templates
assert targets["pro4_terminal"].eq(targets["answer"]).all()

TARGET_PATH = DATA_DIR / "r2_pro4_hint_targets_le50pct.csv"
TARGET_REVIEW_PATH = DATA_DIR / "r2_target_eligibility_audit.csv"
target_columns = [
    "id", "question", "answer", "pro4_hint", "strict_successes",
    "rollout_k", "strict_success_rate", "template_key",
]
targets[target_columns].to_csv(TARGET_PATH, index=False, encoding="utf-8")
train[[
    "id", "answer", "strict_successes", "rollout_k", "strict_success_rate",
    "target_difficulty", "is_holdout", "label_review", "has_verified_pro4",
]].to_csv(TARGET_REVIEW_PATH, index=False, encoding="utf-8")

print("[ROLLOUT COVERAGE]", len(success), "/", len(clean_train))
print("[TARGETS]", len(targets), "questions")
print("[TARGET HIST]")
print(targets.groupby(["rollout_k", "strict_successes"]).size())
print("[EXCLUDED label-review]", int((train["target_difficulty"] & train["label_review"]).sum()))
print("[MISSING/unverified Pro4]", int((train["target_difficulty"] & ~train["has_verified_pro4"]).sum()))
print("[TARGET FILE]", TARGET_PATH)

In [ ]:
# Cell 4 — Load vLLM BF16 base plus frozen Lane B LoRA on A100.
runtime_candidates, nvrtc_candidates = [], []
for site_dir in site.getsitepackages():
    runtime_candidates += glob.glob(str(Path(site_dir) / "nvidia" / "cu13" / "lib" / "libcudart.so.13*"))
    runtime_candidates += glob.glob(str(Path(site_dir) / "nvidia" / "cuda_runtime" / "lib" / "libcudart.so.13*"))
    nvrtc_candidates += glob.glob(str(Path(site_dir) / "nvidia" / "cu13" / "lib" / "libnvrtc.so.13*"))
    nvrtc_candidates += glob.glob(str(Path(site_dir) / "nvidia" / "cuda_nvrtc" / "lib" / "libnvrtc.so.13*"))
runtime_candidates = sorted(set(runtime_candidates))
nvrtc_candidates = sorted(set(nvrtc_candidates))
assert runtime_candidates and nvrtc_candidates, "CUDA 13 libraries missing; rerun Cell 1."
ctypes.CDLL(runtime_candidates[0], mode=ctypes.RTLD_GLOBAL)
ctypes.CDLL(nvrtc_candidates[0], mode=ctypes.RTLD_GLOBAL)
lib_dirs = sorted({str(Path(runtime_candidates[0]).parent), str(Path(nvrtc_candidates[0]).parent)})
old_ld = os.environ.get("LD_LIBRARY_PATH", "")
os.environ["LD_LIBRARY_PATH"] = ":".join(lib_dirs + ([old_ld] if old_ld else []))
os.environ["LIBRARY_PATH"] = os.environ["LD_LIBRARY_PATH"]

check = subprocess.run(
    [sys.executable, "-c", "from vllm.model_executor.models.qwen2 import Qwen2ForCausalLM; print('OK')"],
    env=os.environ.copy(), capture_output=True, text=True,
)
assert check.returncode == 0, check.stderr

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

adapter_config = json.loads(
    (ADAPTER_PATH / "adapter_config.json").read_text(encoding="utf-8")
)
adapter_base = str(adapter_config.get("base_model_name_or_path", "")).rstrip("/")
assert adapter_base == BASE_MODEL or adapter_base.endswith("/Qwen2.5-3B-Instruct"), adapter_base

class _KnownVLLMWarningFilter(logging.Filter):
    _KNOWN = (
        "deprecated support for supporting different tokenizers for different LoRAs",
        "Triton kernel JIT compilation during inference",
    )
    def filter(self, record):
        return not any(part in record.getMessage() for part in self._KNOWN)

known_filter = _KnownVLLMWarningFilter()
for logger_name in ("vllm.v1.engine.input_processor", "vllm.compilation.jit_monitor"):
    logging.getLogger(logger_name).addFilter(known_filter)

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL, revision=MODEL_REVISION, use_fast=True, token=False
)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
torch.backends.cuda.matmul.allow_tf32 = True
for stream, fd in [(sys.stdout, 1), (sys.stderr, 2)]:
    try:
        stream.fileno()
    except Exception:
        stream.fileno = (lambda value=fd: value)

llm = LLM(
    model=BASE_MODEL,
    revision=MODEL_REVISION,
    dtype="bfloat16",
    tensor_parallel_size=1,
    distributed_executor_backend="uni",
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
    max_model_len=MAX_MODEL_LEN,
    max_num_seqs=MAX_NUM_SEQS,
    max_num_batched_tokens=MAX_NUM_BATCHED_TOKENS,
    enable_chunked_prefill=True,
    enable_prefix_caching=True,
    enable_lora=True,
    max_lora_rank=32,
    max_loras=1,
    max_cpu_loras=2,
    performance_mode="throughput",
    trust_remote_code=False,
)
lora_request = LoRARequest("r1_lane_b_checkpoint178", 1, str(ADAPTER_PATH))
print("[VLLM] ready | tokenizer=base | adapter base=", adapter_base)

In [ ]:
# Cell 5 — Qwen rationalization prompt, strict filters, persistence, and k4 warmup.
NUM_RE = re.compile(r"(?<!\d)-?\s*(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.0+)?")

def last_boxed(text):
    idx = str(text).rfind("\\boxed")
    if idx < 0:
        return None
    opening = str(text).find("{", idx)
    if opening < 0:
        return None
    depth = 0
    for pos in range(opening, len(str(text))):
        if str(text)[pos] == "{":
            depth += 1
        elif str(text)[pos] == "}":
            depth -= 1
            if depth == 0:
                return str(text)[opening + 1:pos]
    return None

def extract_integer(text):
    text = str(text or "")
    boxed = last_boxed(text)
    parsed = normalize_integer(boxed) if boxed is not None else None
    if parsed is not None:
        return parsed, "boxed"
    tail = text.rsplit("</think>", 1)[-1]
    marked = re.findall(
        r"(?:final\s+answer|answer|정답)\s*(?:is|:|=)?\s*([^\n.;]*)",
        tail, re.I,
    )
    if marked:
        nums = NUM_RE.findall(marked[-1])
        if nums:
            parsed = normalize_integer(nums[0].replace(" ", ""))
            if parsed is not None:
                return parsed, "final_marker"
    nums = NUM_RE.findall(tail)
    for number in reversed(nums):
        parsed = normalize_integer(number.replace(" ", ""))
        if parsed is not None:
            return parsed, "last_integer_fallback"
    return None, "parse_failure"

def build_sc_prompt(question):
    content = f"{str(question).strip()}\n\n{USER_SUFFIX}"
    return tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": content},
        ],
        tokenize=False, add_generation_prompt=True,
    )

def build_hint_prompt(question, teacher_hint):
    content = (
        f"{str(question).strip()}\n\n"
        "Solve the problem independently. A verified reference derivation is supplied\n"
        "below only as a private mathematical hint. Use it to understand the key idea,\n"
        "but do not mention the hint, the reference, a teacher, or this instruction.\n"
        "Re-derive and verify the solution in your own concise style. End with exactly\n"
        "one final line containing the integer answer as \\boxed{INTEGER}.\n\n"
        "Private reference derivation:\n"
        f"{str(teacher_hint).strip()}"
    )
    return tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": content},
        ],
        tokenize=False, add_generation_prompt=True,
    )

def append_jsonl(path, rows):
    with Path(path).open("a", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
        handle.flush(); os.fsync(handle.fileno())

def load_jsonl_map(path):
    records = {}
    if not Path(path).exists():
        return records
    with Path(path).open(encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                continue
            if row.get("id"):
                records[str(row["id"])] = row
    return records

def path_signature(text):
    text = unicodedata.normalize("NFKC", str(text)).casefold()
    text = re.sub(r"\\boxed\s*\{[^{}]*\}\s*$", "", text).strip()
    text = re.sub(r"\d+(?:\.\d+)?", "#", text)
    text = re.sub(r"[^a-z가-힣#+*/=<>\\]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

REJECT_PATTERNS = {
    "self_contradiction": re.compile(
        r"\b(?:wait|re[- ]?examining|i made (?:an|a) (?:error|mistake)|"
        r"this (?:is|was) (?:wrong|incorrect)|contradiction)\b", re.I
    ),
    "uncertainty": re.compile(
        r"\b(?:cannot determine|not enough information|insufficient information|"
        r"unable to solve|no unique answer|ambiguous)\b", re.I
    ),
    "teacher_meta": re.compile(
        r"\b(?:private reference|reference solution|provided solution|teacher hint|"
        r"the hint says|according to the hint)\b", re.I
    ),
    "unsupported_tool": re.compile(
        r"\b(?:subprocess|requests\.|urllib|read_csv|http://|https://)\b", re.I
    ),
}

# Warm only the production rationalization shape: 64 prompts x k4 = 256 sequences.
for label, count, n, max_tokens, temperature, top_p in [
    ("RATIONALIZE4", MAX_NUM_SEQS // N_RATIONALIZE, N_RATIONALIZE, 4,
     RATIONALIZE_TEMPERATURE, RATIONALIZE_TOP_P),
]:
    prompts = [build_sc_prompt("Compute 1+1.")] * count
    params = SamplingParams(
        n=n, temperature=temperature, top_p=top_p,
        max_tokens=max_tokens, seed=SEED - n,
    )
    started = time.time()
    outputs = llm.generate(prompts, params, lora_request=lora_request, use_tqdm=False)
    assert len(outputs) == count and all(len(request.outputs) == n for request in outputs)
    print(f"[WARMUP {label}] {count} prompts x {n} in {time.time()-started:.1f}s")
    del outputs
torch.cuda.synchronize()

In [ ]:
# Cell 6 — Generate four Lane-B Qwen solutions per target using Pro4 only as a private hint.
RUN_RATIONALIZATION = True
RATIONALIZED_RAW_PATH = CAND_DIR / "r2_pro4_hint_qwen_rollouts_k4.jsonl"

if not RUN_RATIONALIZATION:
    print("[RATIONALIZE] disabled")
else:
    existing_r2 = load_jsonl_map(RATIONALIZED_RAW_PATH)
    pending = targets[~targets["id"].isin(existing_r2)].copy()
    prompt_chunk = max(1, MAX_NUM_SEQS // N_RATIONALIZE)
    print(f"[RATIONALIZE] done={len(existing_r2)} pending={len(pending)} chunk={prompt_chunk}")
    started = time.time()
    for start in range(0, len(pending), prompt_chunk):
        batch = pending.iloc[start:start + prompt_chunk]
        prompts = [
            build_hint_prompt(row.question, row.pro4_hint)
            for row in batch.itertuples(index=False)
        ]
        prompt_lengths = [
            len(tokenizer(prompt, add_special_tokens=False)["input_ids"])
            for prompt in prompts
        ]
        assert max(prompt_lengths) + RATIONALIZE_MAX_NEW_TOKENS <= MAX_MODEL_LEN, (
            max(prompt_lengths), RATIONALIZE_MAX_NEW_TOKENS, MAX_MODEL_LEN
        )
        params = SamplingParams(
            n=N_RATIONALIZE,
            temperature=RATIONALIZE_TEMPERATURE,
            top_p=RATIONALIZE_TOP_P,
            max_tokens=RATIONALIZE_MAX_NEW_TOKENS,
            seed=SEED + start,
        )
        requests = llm.generate(
            prompts, params, lora_request=lora_request, use_tqdm=False
        )
        append_rows = []
        for row, prompt_len, request in zip(batch.itertuples(index=False), prompt_lengths, requests):
            candidates = []
            for idx, completion in enumerate(request.outputs):
                candidates.append({
                    "sample_index": idx,
                    "strict_terminal_answer": terminal_boxed(completion.text),
                    "generated_tokens": len(completion.token_ids),
                    "finish_reason": str(completion.finish_reason or ""),
                    "raw_output": completion.text,
                })
            append_rows.append({
                "id": str(row.id),
                "official_answer": str(row.answer),
                "prior_strict_successes": int(row.strict_successes),
                "prior_rollout_k": int(row.rollout_k),
                "prior_success_rate": float(row.strict_success_rate),
                "pro4_hint_sha256": hashlib.sha256(str(row.pro4_hint).encode()).hexdigest(),
                "prompt_tokens": prompt_len,
                "prompt_version": RATIONALIZE_PROMPT_VERSION,
                "candidates": candidates,
            })
        append_jsonl(RATIONALIZED_RAW_PATH, append_rows)
        existing_r2.update({row["id"]: row for row in append_rows})
        done = len(existing_r2)
        elapsed = time.time() - started
        rate = max(1, done) / max(elapsed, 1e-9)
        eta = (len(targets) - done) / max(rate, 1e-9) / 60
        print(f"[RATIONALIZE] {done}/{len(targets)} eta={eta:.1f}m", flush=True)

    assert set(existing_r2) == set(targets["id"])
    print("[RATIONALIZE COMPLETE]", RATIONALIZED_RAW_PATH)
    print("[RAW SHA]", sha256_file(RATIONALIZED_RAW_PATH))

In [ ]:
# Cell 7 — Filter Qwen outputs, add untouched R1 anchors, and save the R2 mix.
assert RUN_RATIONALIZATION, "Run Cell 6 first."
raw_r2 = load_jsonl_map(RATIONALIZED_RAW_PATH)
target_by_id = targets.set_index("id").to_dict("index")
audit_rows, selected_rows = [], []
reject_counter = Counter()

for qid in targets["id"]:
    metadata = target_by_id[qid]
    record = raw_r2[qid]
    official = str(metadata["answer"])
    teacher_hint = str(metadata["pro4_hint"])
    teacher_norm = re.sub(r"\s+", " ", normalize_question(teacher_hint))
    accepted, seen_signatures = [], set()

    for candidate in record["candidates"]:
        raw = str(candidate["raw_output"])
        terminal = terminal_boxed(raw)
        word_count = len(raw.split())
        training_prompt = build_sc_prompt(metadata["question"])
        total_tokens = len(tokenizer(
            training_prompt + raw, add_special_tokens=False
        )["input_ids"])
        signature = path_signature(raw)
        raw_norm = re.sub(r"\s+", " ", normalize_question(raw))
        similarity = difflib.SequenceMatcher(
            None, raw_norm[:12000], teacher_norm[:12000]
        ).ratio()
        reasons = []
        if terminal != official:
            reasons.append("terminal_boxed_answer_mismatch")
        if word_count < 20:
            reasons.append("solution_too_short")
        if word_count > MAX_VISIBLE_WORDS:
            reasons.append("solution_too_long")
        if total_tokens > MAX_SFT_TOKENS:
            reasons.append("total_tokens_gt_2048")
        if candidate.get("finish_reason") == "length":
            reasons.append("generation_hit_4096_cap")
        if similarity >= 0.98:
            reasons.append("near_verbatim_teacher_copy")
        for reason, pattern in REJECT_PATTERNS.items():
            if pattern.search(raw):
                reasons.append(reason)
        if not signature:
            reasons.append("empty_path_signature")
        elif signature in seen_signatures:
            reasons.append("duplicate_reasoning_path")

        decision = "accept" if not reasons else "reject"
        for reason in reasons:
            reject_counter[reason] += 1
        audit_rows.append({
            "id": qid,
            "sample_index": candidate["sample_index"],
            "official_answer": official,
            "prior_strict_successes": metadata["strict_successes"],
            "prior_rollout_k": metadata["rollout_k"],
            "terminal_boxed_answer": terminal or "",
            "word_count": word_count,
            "total_tokens": total_tokens,
            "teacher_similarity": similarity,
            "decision": decision,
            "reasons": " | ".join(reasons),
        })
        if decision == "accept":
            seen_signatures.add(signature)
            accepted.append({
                "id": qid,
                "question": metadata["question"],
                "answer": official,
                "solution": raw,
                "source": "lane_b_qwen_r2_pro4_private_hint",
                "origin": f"prior_{int(metadata['strict_successes'])}_of_{int(metadata['rollout_k'])}",
                "prior_success_rate": metadata["strict_success_rate"],
                "total_tokens": total_tokens,
                "sample_index": candidate["sample_index"],
            })

    accepted.sort(key=lambda item: (item["total_tokens"], item["sample_index"]))
    # 0-success examples are riskiest; keep one. Other <=50% rows may keep two.
    max_paths = 1 if int(metadata["strict_successes"]) == 0 else 2
    selected_rows.extend(accepted[:max_paths])

audit = pd.DataFrame(audit_rows)
rationalized = pd.DataFrame(selected_rows)
assert len(rationalized), "No rationalized rows survived; inspect audit before training."
assert rationalized.groupby("id").size().max() <= 2
assert all(
    terminal_boxed(solution) == answer
    for solution, answer in zip(rationalized["solution"], rationalized["answer"])
)
assert not set(rationalized["id"]) & holdout_ids

AUDIT_PATH = DATA_DIR / "r2_qwen_pro4_hint_candidate_audit.csv"
RATIONALIZED_PATH = DATA_DIR / "r2_qwen_pro4_hint_verified.csv"
audit.to_csv(AUDIT_PATH, index=False, encoding="utf-8")
rationalized.to_csv(RATIONALIZED_PATH, index=False, encoding="utf-8")

r1 = read_csv_strings(R1_VERIFIED_PATH)
assert {"id", "question", "answer", "solution", "source", "origin_shard"}.issubset(r1.columns)
r1["answer"] = r1["answer"].map(normalize_integer)
r1["template_key"] = r1["question"].map(template_key)
r1 = r1[
    ~r1["id"].isin(set(targets["id"]))
    & ~r1["id"].isin(holdout_ids)
    & ~r1["template_key"].isin(holdout_templates)
].copy()
r1["hash_rank"] = r1["id"].map(
    lambda qid: hashlib.sha256(f"{SEED}|anchor|{qid}".encode()).hexdigest()
)
r1 = r1.sort_values(["hash_rank", "id"]).drop_duplicates("id", keep="first")
anchor_needed = math.ceil(len(rationalized) * (1 - TARGET_MIX_SHARE) / TARGET_MIX_SHARE)
anchors = r1.head(min(anchor_needed, len(r1))).copy()
anchors = anchors[["id", "question", "answer", "solution"]]
anchors["source"] = "r1_native_untouched_anchor"
anchors["origin"] = "r1_non_target_anchor"
anchors["prior_success_rate"] = ""
anchors["total_tokens"] = ""
anchors["sample_index"] = ""

output_columns = [
    "id", "question", "answer", "solution", "source", "origin",
    "prior_success_rate", "total_tokens", "sample_index",
]
mix = pd.concat([
    rationalized[output_columns], anchors[output_columns]
], ignore_index=True)
mix["sort_key"] = mix.apply(
    lambda row: hashlib.sha256(f"{SEED}|mix|{row['id']}|{row['source']}".encode()).hexdigest(),
    axis=1,
)
mix = mix.sort_values("sort_key").drop(columns="sort_key").reset_index(drop=True)
assert not set(mix["id"]) & holdout_ids
assert not set(mix["question"].map(template_key)) & holdout_templates
assert mix["answer"].str.fullmatch(r"-?\d+").all()
assert all(terminal_boxed(solution) == answer for solution, answer in zip(mix["solution"], mix["answer"]))

MIX_PATH = DATA_DIR / "r2_pro4_hint_qwen_65_anchor35_train.csv"
mix.to_csv(MIX_PATH, index=False, encoding="utf-8")
filter_report = {
    "run_id": RUN_ID,
    "target_questions": len(targets),
    "target_success_definition": "prior strict success rate <= 0.50 (<=2/4; reused pilot <=4/8)",
    "questions_with_verified_qwen_trace": int(rationalized["id"].nunique()),
    "verified_qwen_traces": len(rationalized),
    "target_question_coverage": float(rationalized["id"].nunique() / len(targets)),
    "anchor_rows": len(anchors),
    "final_mix_rows": len(mix),
    "actual_target_trace_share": float(len(rationalized) / len(mix)),
    "reject_reason_counts": dict(reject_counter),
    "artifacts": {
        "targets": str(TARGET_PATH),
        "raw_qwen_rollouts": str(RATIONALIZED_RAW_PATH),
        "candidate_audit": str(AUDIT_PATH),
        "rationalized_verified": str(RATIONALIZED_PATH),
        "training_mix": str(MIX_PATH),
        "training_mix_sha256": sha256_file(MIX_PATH),
    },
    "official_evaluation_files_read": [],
    "commercial_api_calls": 0,
}
FILTER_REPORT_PATH = REPORT_DIR / "r2_data_generation_report.json"
FILTER_REPORT_PATH.write_text(
    json.dumps(filter_report, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(json.dumps(filter_report, ensure_ascii=False, indent=2))
print("[R2 TRAINING MIX]", MIX_PATH)

In [ ]:
# Cell 8 — Immutable generation report and optional runtime release.
assert RUN_RATIONALIZATION
final_report = {
    "run_id": RUN_ID,
    "status": "completed",
    "objective": "Generate Lane-B Qwen rationalizations for <=50%-success official-train questions using existing verified Pro4 solutions only as private hints.",
    "base_model": BASE_MODEL,
    "model_revision": MODEL_REVISION,
    "adapter": str(ADAPTER_PATH),
    "adapter_weight_sha256": sha256_file(adapter_weight),
    "prompt_version": RATIONALIZE_PROMPT_VERSION,
    "generation": {
        "n": N_RATIONALIZE,
        "temperature": RATIONALIZE_TEMPERATURE,
        "top_p": RATIONALIZE_TOP_P,
        "max_new_tokens": RATIONALIZE_MAX_NEW_TOKENS,
        "max_model_len": MAX_MODEL_LEN,
    },
    "r2_data": filter_report,
    "input_hashes": {
        "clean_train": sha256_file(CLEAN_TRAIN_PATH),
        "holdout_tune_ids": sha256_file(TUNE_PATH),
        "holdout_dev_ids": sha256_file(DEV_PATH),
        "holdout_test_ids": sha256_file(TEST_HOLDOUT_PATH),
        "r1_verified": sha256_file(R1_VERIFIED_PATH),
        "r1_manifest": sha256_file(R1_MANIFEST_PATH),
        "pro4_verified": sha256_file(PRO4_VERIFIED_PATH),
    },
    "safety": {
        "leaderboard_or_competition_test_read": False,
        "external_api_calls": 0,
        "pro4_usage": "existing verified official-train solution used as private generation hint only",
        "pro4_text_used_as_sft_response": False,
        "validation_ids_or_templates_in_training_mix": False,
    },
}
FINAL_REPORT_PATH = REPORT_DIR / "experiment_report.json"
FINAL_REPORT_PATH.write_text(
    json.dumps(final_report, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("[FINAL REPORT]", FINAL_REPORT_PATH)
print("[R2 TRAINING MIX]", MIX_PATH)
print("[NEXT] Audit the generated corpus before running any R2 training.")

DISCONNECT_RUNTIME = False
if DISCONNECT_RUNTIME:
    from google.colab import runtime
    runtime.unassign()
else:
    print("[RUNTIME] retained")